# 🔬 ABLATION STUDY MiT-B0 CHUẨN SOTA (`16,000 iters × bs=8`, `lr=6e-5`, `PolyLR`) TRÊN COLAB G4

> **Khác biệt cốt lõi so với bản `5k × bs=32` (Tab 1):**
> Bản này dùng đúng **công thức siêu tham số đã chứng minh hiệu quả trên RTM** (giống `MiT-B2 SOTA 19.91%`, `B0 Mobile`, và paper ASC-Former gốc):
> * **Số bước cập nhật trọng số:** **`16,000 iters`** (đủ 16k gradient updates như bản Ablation cũ, tăng 3.2× so với `5k`!)
> * **Batch size & Lượng mẫu:** **`batch_size = 8`** → **`16,000 × 8 = 128,000` crops** (gấp **4×** bản Ablation cũ `32k` crops, không bị tụt Precision hay `SupConLoss` lấn át như `bs=32`!)
> * **Tối ưu & Điều chuẩn:** **`AdamW lr=6e-5` + `Warmup 1,000` + `PolyLR`**, `drop_path_rate=0.1`, Native **BF16 + TF32**, Full-Scan **`stride=384×384`** chuẩn lưới `8×8` DCT trên toàn bộ **1,203 ảnh `val.txt`**.
>
> **4 mô hình độc lập (`One-Factor-At-a-Time`, `seed=3407`):**
> * **`M1_Baseline_2Stream`:** `RGB + Forensic`, `RandomCrop`, `ContrastiveHeadV2` (Memory Bank).
> * **`M2_RealOCR_3Stream`:** `M1` + nhánh **OCR thật** (`TriCMNeXt`, fix đường dẫn `ocr_spatial` + đồng bộ flip H/V).
> * **`M3_Focused_Rotate90`:** `M1` + `FocusedCrop(pos_prob=0.65)` + `RandomRotate90(0.5)`.
> * **`M4_LiteProto_CE`:** `M1` + **`LitePrototypeHead`** (`128ch/64dim`, DW-Sep, giảm **97%** FLOPs Head xuống `5.90 GFLOPs`) + Dual EMA Prototypes + **`CE` thuần** *(đã bỏ per-image `DiceLoss`)*.
>
> **Chạy song song CẢ 4 MODEL CÙNG LÚC 1 LƯỢT trên G4 96GB (`~38 GB / 96 GB VRAM`) & tách riêng hoàn toàn thư mục Drive (`/content/drive/MyDrive/RTM_Ablation_B0_16k_bs8_G4`) để chạy song song an toàn với Tab Colab 1!**

### 🟢 BƯỚC 1: KẾT NỐI DRIVE, CÀI MÔI TRƯỜNG & GIẢI NÉN DATASET (~3 Phút)

In [2]:
import os, sys, glob, torch
from google.colab import drive

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"🖥️ GPU: {props.name} | VRAM: {props.total_memory/(1024**3):.1f} GB | BF16: {torch.cuda.is_bf16_supported()}")
    print(f"   CPU cores: {os.cpu_count()} | PyTorch: {torch.__version__}")
else:
    raise RuntimeError("❌ Chưa bật GPU Runtime! Hãy chọn Runtime -> Change runtime type -> G4 GPU.")

drive.mount("/content/drive")

%cd /content
!rm -rf TMD
!git clone https://github.com/gagaduy/TMD.git

sys.path.insert(0, "/content/TMD/ASCFormer")
%env PYTHONPATH=/content/TMD/ASCFormer

!pip install -q yapf==0.40.1 mmcv-lite==2.0.0 mmengine==0.7.0 timm einops prettytable rich onnx onnxruntime gdown

# Link chia sẻ RealTextMan.zip (tự tải bằng gdown nếu nick Colab hiện tại chưa có file trên MyDrive)
GDRIVE_ZIP_URL = "https://drive.google.com/file/d/1uBtDzuTcqAFX9vWUH3n8Vn3EewWpW5kP/view?usp=drivesdk"

os.makedirs("/content/TMD/ASCFormer/data/ttd", exist_ok=True)
drive_zip = "/content/drive/MyDrive/RealTextMan.zip"
if not os.path.exists(drive_zip):
    matches = glob.glob("/content/drive/MyDrive/**/RealTextMan.zip", recursive=True)
    if matches:
        drive_zip = matches[0]
    elif GDRIVE_ZIP_URL.strip():
        drive_zip = "/content/RealTextMan.zip"
        print("🌐 Không thấy trên MyDrive, đang tải RealTextMan.zip từ link chia sẻ bằng gdown...")
        !gdown --fuzzy "{GDRIVE_ZIP_URL.strip()}" -O "{drive_zip}"
    else:
        raise FileNotFoundError("❌ Không thấy RealTextMan.zip trên MyDrive và chưa có GDRIVE_ZIP_URL!")

print(f"📦 Đang giải nén {drive_zip} sang SSD Colab...")
!unzip -q -o "{drive_zip}" -d /content/TMD/ASCFormer/data/ttd/

ocr_cnt = len(glob.glob("/content/TMD/ASCFormer/data/ttd/RealTextMan/ocr_spatial/*.png"))
img_cnt = len(glob.glob("/content/TMD/ASCFormer/data/ttd/RealTextMan/JPEGImages/*.jpg"))
print(f"✅ Dataset sẵn sàng: {img_cnt} ảnh JPEGImages, {ocr_cnt} bản đồ ocr_spatial thật!")
assert img_cnt > 0 and ocr_cnt > 0

🖥️ GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM: 95.0 GB | BF16: True
   CPU cores: 48 | PyTorch: 2.14.1+cu130
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
Cloning into 'TMD'...
remote: Enumerating objects: 1890, done.
remote: Counting objects: 100% (1890/1890), done.
remote: Compressing objects: 100% (930/930), done.
remote: Total 1890 (delta 1074), reused 1755 (delta 939), pack-reused 0 (from 0)
Receiving objects: 100% (1890/1890), 14.27 MiB | 46.67 MiB/s, done.
Resolving deltas: 100% (1074/1074), done.
env: PYTHONPATH=/content/TMD/ASCFormer
🌐 Không thấy trên MyDrive, đang tải RealTextMan.zip từ link chia sẻ bằng gdown...
Downloading...
From (original): https://drive.google.com/uc?id=1uBtDzuTcqAFX9vWUH3n8Vn3EewWpW5kP
From (redirected): https://drive.google.com/uc?id=1uBtDzuTcqAFX9vWUH3n8Vn3EewWpW5kP&confirm=t&uuid=93dd85c1-f338-4be8-b8c2-d3716fc8dec0
To: /content/RealTextMan.zip
1

### 🔥 BƯỚC 2: HUẤN LUYỆN SONG SONG CẢ 4 MÔ HÌNH `M1 → M4` (`16,000 iters × bs=8`, `lr=6e-5`, `PolyLR`)
> * **Tách biệt hoàn toàn với Tab 1:** lưu riêng tại `/content/drive/MyDrive/RTM_Ablation_B0_16k_bs8_G4/`.
> * Trên G4 96GB, `bs=8` chỉ ăn ~5–12 GB/model nên chạy **song song cả 4 model cùng lúc 1 lượt** (`MAX_PARALLEL_TRAIN = 4`, tổng ~38 GB VRAM!).
> * **Tự động backup checkpoint trung gian (`4k, 8k, 12k, best_mIoU`)** lên Drive trong lúc đang train và tự `--resume` nếu ngắt phiên.
> * **Ước tính tổng thời gian trên G4:** khoảng **~75 – 95 phút** cho cả 4 model cùng lúc!

In [ ]:
import os, sys, glob, time, re, shutil, subprocess, torch

%cd /content/TMD
!git pull origin main
%cd /content/TMD/ASCFormer
%env PYTHONPATH=/content/TMD/ASCFormer

# Tải trước Pretrained MiT-B0 để tránh race condition khi chạy 4 tiến trình song song
_ = torch.hub.load_state_dict_from_url(
    "https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b0_20220624-7e0fe6dd.pth",
    map_location="cpu"
)

DRIVE_ROOT = "/content/drive/MyDrive/RTM_Ablation_B0_16k_bs8_G4"
LOCAL_ROOT = "/content/work_dirs/ablation_b0_16k_bs8"
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(LOCAL_ROOT, exist_ok=True)

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
MAX_PARALLEL_TRAIN = 4 if vram_gb >= 60 else (2 if vram_gb >= 30 else 1)
print(f"⚡ G4 Scheduler (16k x bs=8): VRAM={vram_gb:.1f} GB -> MAX_PARALLEL_TRAIN = {MAX_PARALLEL_TRAIN}")

ABLATION_SUITE = [
    ("M1_Baseline_2Stream",   "configs/ascformer/ablation_b0_16k_bs8_m1_baseline.py"),
    ("M2_RealOCR_3Stream",    "configs/ascformer/ablation_b0_16k_bs8_m2_ocr.py"),
    ("M3_Focused_Rotate90",   "configs/ascformer/ablation_b0_16k_bs8_m3_focused.py"),
    ("M4_LiteProto_CE",   "configs/ascformer/ablation_b0_16k_bs8_m4_liteproto.py"),
]

def backup_to_drive(local_dir, drive_dir, synced_set=None):
    os.makedirs(drive_dir, exist_ok=True)
    for ext in ("*.pth", "*.log", "*.json", "*.py", "*.png"):
        for fp in glob.glob(os.path.join(local_dir, ext)):
            bn = os.path.basename(fp)
            if synced_set is not None and bn.endswith(".pth"):
                sig = (bn, os.path.getsize(fp), int(os.path.getmtime(fp)))
                if sig in synced_set:
                    continue
                shutil.copy2(fp, os.path.join(drive_dir, bn))
                synced_set.add(sig)
            else:
                shutil.copy2(fp, os.path.join(drive_dir, bn))
    for fp in glob.glob(os.path.join(local_dir, "*", "*.log")) + glob.glob(os.path.join(local_dir, "*", "vis_data", "*.json")):
        shutil.copy2(fp, os.path.join(drive_dir, os.path.basename(fp)))
    for fp in glob.glob(os.path.join(local_dir, "**", "confusion_matrix.png"), recursive=True):
        shutil.copy2(fp, os.path.join(drive_dir, "confusion_matrix.png"))

def parse_progress(log_path):
    if not os.path.exists(log_path):
        return "Đang khởi tạo..."
    try:
        with open(log_path, "r", errors="ignore") as f:
            lines = f.readlines()[-40:]
        for ln in reversed(lines):
            if "Iter(train)" in ln:
                m = re.search(r"Iter\(train\)\s*\[\s*(\d+/\d+)\].*?eta:\s*([0-9:]+).*?loss:\s*([0-9.]+)", ln)
                if m:
                    return f"Iter [{m.group(1)}] | loss={m.group(3)} | ETA={m.group(2)}"
            if "Iter(val)" in ln:
                return "Đang chạy Valid 1,203 ảnh..."
    except Exception:
        pass
    return "Đang nạp dữ liệu / compile..."

queue = []
for name, cfg_path in ABLATION_SUITE:
    d_dir = os.path.join(DRIVE_ROOT, name)
    if os.path.exists(os.path.join(d_dir, "iter_16000.pth")):
        print(f"✅ [{name}] Đã hoàn tất 16,000 iters trên Drive — bỏ qua!")
    else:
        queue.append((name, cfg_path))

running = {}
synced_map = {}
failed = []
t_global = time.time()

# Tạo worker launcher sạch (vá yapf + torch.load PyTorch 2.6+ + TF32)
WORKER_PY = "/tmp/run_train_worker_16k.py"
with open(WORKER_PY, "w") as wf:
    wf.write("""import os, sys, runpy, torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(*args, **kwargs)
torch.load = _safe_load
try:
    import yapf.yapflib.yapf_api as _y
    import mmengine.config.config as _m
    _orig_fc = _y.FormatCode
    def _safe_fc(*args, **kwargs):
        kwargs.pop('verify', None)
        return _orig_fc(*args, **kwargs)
    _y.FormatCode = _safe_fc
    _m.FormatCode = _safe_fc
except Exception:
    pass
sys.argv = ['tools/train.py'] + sys.argv[1:]
runpy.run_path('tools/train.py', run_name='__main__')
""")

while queue or running:
    while queue and len(running) < MAX_PARALLEL_TRAIN:
        name, cfg_path = queue.pop(0)
        l_dir = os.path.join(LOCAL_ROOT, name)
        d_dir = os.path.join(DRIVE_ROOT, name)
        os.makedirs(l_dir, exist_ok=True)
        os.makedirs(d_dir, exist_ok=True)

        d_ckpts = sorted(glob.glob(os.path.join(d_dir, "iter_*.pth")), key=os.path.getmtime)
        resume_args = []
        if d_ckpts:
            dst = os.path.join(l_dir, os.path.basename(d_ckpts[-1]))
            shutil.copy2(d_ckpts[-1], dst)
            resume_args = ["--resume", dst]

        log_path = os.path.join(l_dir, f"{name}_stdout.log")
        fh = open(log_path, "w")

        cmd = [sys.executable, WORKER_PY, cfg_path, "--work-dir", l_dir] + resume_args
        env = os.environ.copy()
        env["PYTHONPATH"] = "/content/TMD/ASCFormer"
        env["OMP_NUM_THREADS"] = "2"
        proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env)
        running[name] = (proc, fh, log_path, l_dir, d_dir, time.time())
        synced_map[name] = set()
        print(f"🚀 Khởi chạy [{name}] (PID={proc.pid}) | Resume={resume_args}")

    time.sleep(30)
    elapsed = (time.time() - t_global) / 60.0
    gpu_stat = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"\n⏱️ [{elapsed:.1f}p] G4 GPU (VRAM / Util): {gpu_stat}")

    finished_names = []
    for name, (proc, fh, log_path, l_dir, d_dir, t0) in running.items():
        ret = proc.poll()
        if ret is None:
            print(f"   🔄 {name:<22}: {parse_progress(log_path)}")
            # Đồng bộ ngay checkpoint mới (4k/8k/12k/best) lên Drive để không mất tiến độ nếu ngắt phiên
            backup_to_drive(l_dir, d_dir, synced_set=synced_map[name])
        else:
            fh.close()
            dt = (time.time() - t0) / 60.0
            if ret == 0:
                print(f"   🎉 {name:<22}: XONG 16,000 iters trong {dt:.1f}p! Đang sao lưu đầy đủ lên Drive...")
                backup_to_drive(l_dir, d_dir, synced_set=None)
            else:
                print(f"   ❌ {name:<22}: LỖI (exit={ret})! 30 dòng log cuối:")
                with open(log_path, "r", errors="ignore") as lf:
                    print("".join(lf.readlines()[-30:]))
                failed.append(name)
            finished_names.append(name)
    for n in finished_names:
        running.pop(n)

assert not failed, f"Có model bị lỗi: {failed}"
print(f"\n🏆 HOÀN TẤT CẢ 4 MÔ HÌNH M1 → M4 (16k × bs=8) TRONG {(time.time()-t_global)/60.0:.1f} PHÚT!")

/content/TMD
From https://github.com/gagaduy/TMD
 * branch            main       -> FETCH_HEAD
Already up to date.
/content/TMD/ASCFormer
env: PYTHONPATH=/content/TMD/ASCFormer
Downloading: "https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b0_20220624-7e0fe6dd.pth" to /root/.cache/torch/hub/checkpoints/mit_b0_20220624-7e0fe6dd.pth


100%|██████████| 12.7M/12.7M [00:00<00:00, 57.1MB/s]


⚡ G4 Scheduler (16k x bs=8): VRAM=95.0 GB -> MAX_PARALLEL_TRAIN = 4
🚀 Khởi chạy [M1_Baseline_2Stream] (PID=3880) | Resume=[]
🚀 Khởi chạy [M2_RealOCR_3Stream] (PID=3881) | Resume=[]
🚀 Khởi chạy [M3_Focused_Rotate90] (PID=3882) | Resume=[]
🚀 Khởi chạy [M4_LiteProto_CE] (PID=3883) | Resume=[]

⏱️ [0.5p] G4 GPU (VRAM / Util): 12599 MiB, 97887 MiB, 0 %
   🔄 M1_Baseline_2Stream   : Đang nạp dữ liệu / compile...
   🔄 M2_RealOCR_3Stream    : Đang nạp dữ liệu / compile...
   🔄 M3_Focused_Rotate90   : Đang nạp dữ liệu / compile...
   🔄 M4_LiteProto_CE       : Đang nạp dữ liệu / compile...

⏱️ [1.0p] G4 GPU (VRAM / Util): 19735 MiB, 97887 MiB, 0 %
   🔄 M1_Baseline_2Stream   : Đang nạp dữ liệu / compile...
   🔄 M2_RealOCR_3Stream    : Đang nạp dữ liệu / compile...
   🔄 M3_Focused_Rotate90   : Đang nạp dữ liệu / compile...
   🔄 M4_LiteProto_CE       : Đang nạp dữ liệu / compile...

⏱️ [1.5p] G4 GPU (VRAM / Util): 19139 MiB, 97887 MiB, 0 %
   🔄 M1_Baseline_2Stream   : Đang nạp dữ liệu / compile...
 

### 📊 BƯỚC 3: QUÉT ĐA NGƯỠNG FULL-SCAN SONG SONG CẢ 4 MÔ HÌNH (`M1 → M4`, 1,203 ẢNH VAL) & IN BẢNG SO SÁNH
> Chạy song song cả 4 tiến trình sweep cùng lúc trên G4 (~5–6 phút xong cả 4 model!), quét mịn từ `0.03 → 0.60` và lưu đầy đủ `.json`, `.txt`, `.md` lên `/content/drive/MyDrive/RTM_Ablation_B0_16k_bs8_G4/`.

In [ ]:
import os, sys, glob, json, time, subprocess
from prettytable import PrettyTable

%cd /content/TMD/ASCFormer
%env PYTHONPATH=/content/TMD/ASCFormer

DRIVE_ROOT = "/content/drive/MyDrive/RTM_Ablation_B0_16k_bs8_G4"
LOCAL_ROOT = "/content/work_dirs/ablation_b0_16k_bs8"
ABLATION_SUITE = [
    ("M1_Baseline_2Stream",   "configs/ascformer/ablation_b0_16k_bs8_m1_baseline.py"),
    ("M2_RealOCR_3Stream",    "configs/ascformer/ablation_b0_16k_bs8_m2_ocr.py"),
    ("M3_Focused_Rotate90",   "configs/ascformer/ablation_b0_16k_bs8_m3_focused.py"),
    ("M4_LiteProto_CE",   "configs/ascformer/ablation_b0_16k_bs8_m4_liteproto.py"),
]
THRESHOLDS = [
    "0.03", "0.04", "0.05", "0.06", "0.08",
    "0.10", "0.15", "0.20", "0.25", "0.30",
    "0.35", "0.40", "0.45", "0.50", "0.55", "0.60"
]

MAX_PARALLEL_SWEEP = 4
jobs = []
for name, cfg_path in ABLATION_SUITE:
    d_dir = os.path.join(DRIVE_ROOT, name)
    l_dir = os.path.join(LOCAL_ROOT, name)
    sweep_json = os.path.join(d_dir, f"{name}_sweep.json")
    if os.path.exists(sweep_json):
        print(f"✅ [{name}] Đã có file sweep JSON — bỏ qua!")
        continue
    # Ưu tiên iter_16000.pth (hoặc best_mIoU nếu > 0.50 đạt cao hơn)
    cands = sorted(glob.glob(f"{l_dir}/iter_16000.pth") + glob.glob(f"{d_dir}/iter_16000.pth"), key=os.path.getmtime)
    if not cands:
        cands = sorted(glob.glob(f"{l_dir}/best_mIoU_*.pth") + glob.glob(f"{d_dir}/best_mIoU_*.pth"), key=os.path.getmtime)
    assert cands, f"❌ Không tìm thấy checkpoint cho {name} trong {l_dir} hay {d_dir}!"
    ckpt = cands[-1]
    jobs.append((name, cfg_path, ckpt, sweep_json))

SWEEP_WORKER_PY = "/tmp/run_sweep_worker_16k.py"
with open(SWEEP_WORKER_PY, "w") as wf:
    wf.write("""import sys, runpy, torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(*args, **kwargs)
torch.load = _safe_load
try:
    import yapf.yapflib.yapf_api as _y
    import mmengine.config.config as _m
    _orig_fc = _y.FormatCode
    def _safe_fc(*args, **kwargs):
        kwargs.pop('verify', None)
        return _orig_fc(*args, **kwargs)
    _y.FormatCode = _safe_fc
    _m.FormatCode = _safe_fc
except Exception:
    pass
sys.argv = ['tools/test_threshold_sweep.py'] + sys.argv[1:]
runpy.run_path('tools/test_threshold_sweep.py', run_name='__main__')
""")

running = {}
t0 = time.time()
while jobs or running:
    while jobs and len(running) < MAX_PARALLEL_SWEEP:
        name, cfg_path, ckpt, sweep_json = jobs.pop(0)
        sweep_wd = os.path.join(LOCAL_ROOT, name, "sweep_work")
        cmd = [sys.executable, SWEEP_WORKER_PY, cfg_path, ckpt, "--ann-file", "val.txt", "--work-dir", sweep_wd, "--thresholds", *THRESHOLDS, "--out-json", sweep_json]
        env = os.environ.copy()
        env["PYTHONPATH"] = "/content/TMD/ASCFormer"
        sweep_log_path = os.path.join(os.path.dirname(sweep_json), f"{name}_sweep.log")
        log_f = open(sweep_log_path, "w")
        p = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=env)
        running[name] = (p, log_f, sweep_log_path)
        print(f"🔍 Bắt đầu Full-Scan Sweep 1,203 ảnh: [{name}] ({os.path.basename(ckpt)})")
    time.sleep(15)
    done = []
    for name, (p, log_f, sweep_log_path) in running.items():
        ret = p.poll()
        if ret is not None:
            log_f.close()
            if ret != 0:
                print(f"❌ Sweep lỗi cho {name}! 30 dòng log cuối:")
                with open(sweep_log_path, "r", errors="ignore") as lf:
                    print("".join(lf.readlines()[-30:]))
                raise RuntimeError(f"Sweep failed for {name}")
            print(f"   ✅ Xong Sweep [{name}]!")
            done.append(name)
    for n in done:
        running.pop(n)

print(f"⏱️ Hoàn tất toàn bộ 4 Sweep trong {(time.time()-t0)/60.0:.1f} phút!")

summary = {}
t_std = PrettyTable(["Model", "tau", "Tamper IoU (%)", "mIoU (%)", "Precision (%)", "Recall (%)", "F1 / Dice (%)"])
t_opt = PrettyTable(["Model", "Peak tau*", "Tamper IoU (%)", "mIoU (%)", "Precision (%)", "Recall (%)", "F1 / Dice (%)"])

md_lines = [
    "# Kết quả Controlled Ablation Study MiT-B0 (M1–M4) — 16k × bs=8 trên Colab G4\n",
    "- Giao thức chuẩn SOTA: `16,000 iters x batch_size 8 = 128,000 crops`, `AdamW lr=6e-5`, `PolyLR`, `drop_path=0.1`, BF16 + TF32, Full-Scan `stride=384` trên 1,203 ảnh `val.txt`.\n",
    "## 1. Ngưỡng chuẩn (tau = 0.500)\n",
    "| Model | Tamper IoU (%) | mIoU (%) | Precision (%) | Recall (%) | F1 / Dice (%) |",
    "| :--- | :---: | :---: | :---: | :---: | :---: |",
]
md_opt_lines = [
    "\n## 2. Ngưỡng tối ưu (Peak tau*)\n",
    "| Model | Peak tau* | Tamper IoU (%) | mIoU (%) | Precision (%) | Recall (%) | F1 / Dice (%) |",
    "| :--- | :---: | :---: | :---: | :---: | :---: | :---: |",
]

for name, _ in ABLATION_SUITE:
    sweep_json = os.path.join(DRIVE_ROOT, name, f"{name}_sweep.json")
    res = json.load(open(sweep_json))["results"]
    r50 = [r for r in res if abs(r["threshold"] - 0.50) < 1e-4][0]
    rpk = max(res, key=lambda r: r["tampered_iou"])
    summary[name] = {"tau_050": r50, "peak": rpk}
    t_std.add_row([name, "0.500", f"{r50['tampered_iou']:.2f}", f"{r50['m_iou']:.2f}", f"{r50['tampered_precision']:.2f}", f"{r50['tampered_recall']:.2f}", f"{r50['tampered_dice']:.2f}"])
    t_opt.add_row([name, f"{rpk['threshold']:.3f}", f"{rpk['tampered_iou']:.2f}", f"{rpk['m_iou']:.2f}", f"{rpk['tampered_precision']:.2f}", f"{rpk['tampered_recall']:.2f}", f"{rpk['tampered_dice']:.2f}"])
    md_lines.append(f"| {name} | {r50['tampered_iou']:.2f} | {r50['m_iou']:.2f} | {r50['tampered_precision']:.2f} | {r50['tampered_recall']:.2f} | {r50['tampered_dice']:.2f} |")
    md_opt_lines.append(f"| {name} | {rpk['threshold']:.3f} | {rpk['tampered_iou']:.2f} | {rpk['m_iou']:.2f} | {rpk['tampered_precision']:.2f} | {rpk['tampered_recall']:.2f} | {rpk['tampered_dice']:.2f} |")

sum_path = os.path.join(DRIVE_ROOT, "ablation_16k_bs8_M1_to_M4_summary.json")
txt_path = os.path.join(DRIVE_ROOT, "ablation_16k_bs8_M1_to_M4_report.txt")
md_path  = os.path.join(DRIVE_ROOT, "ablation_16k_bs8_M1_to_M4_report.md")

with open(sum_path, "w") as f:
    json.dump(summary, f, indent=2)
with open(txt_path, "w") as f:
    f.write("BẢNG ABLATION ĐỘC LẬP M1-M4 (16k x bs=8) TẠI NGƯỠNG CHUẨN (tau = 0.500)\n")
    f.write(str(t_std) + "\n\n")
    f.write("BẢNG ABLATION ĐỘC LẬP M1-M4 (16k x bs=8) TẠI NGƯỠNG TỐI ƯU (Peak tau*)\n")
    f.write(str(t_opt) + "\n")
with open(md_path, "w") as f:
    f.write("\n".join(md_lines + md_opt_lines) + "\n")

print("\n📌 BẢNG ABLATION ĐỘC LẬP M1–M4 (16k × bs=8) TẠI NGƯỠNG CHUẨN (tau = 0.500):")
print(t_std)
print("\n📌 BẢNG ABLATION ĐỘC LẬP M1–M4 (16k × bs=8) TẠI NGƯỠNG TỐI ƯU (Peak tau*):")
print(t_opt)
print("💾 Đã lưu đầy đủ trên Google Drive:")
print("   - Báo cáo bảng dễ đọc :", txt_path)
print("   - Báo cáo Markdown    :", md_path)
print("   - Dữ liệu JSON chuẩn  :", sum_path)
print("💡 Mẹo: Bấm Ctrl+S (File -> Save a copy in Drive) để lưu luôn bản notebook kèm output nhé!")